In [1]:
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [2]:
results = pd.read_csv("../data/processed/anomaly_results.csv")
insiders = pd.read_csv("../data/raw/answers/insiders.csv")

print("Model results:", results.shape)
print("Insider records:", insiders.shape)

Model results: (330452, 10)
Insider records: (191, 6)


In [4]:
insiders = insiders[insiders["dataset"] == 4.2].copy()

insiders["start"] = pd.to_datetime(
    insiders["start"], format="%m/%d/%Y %H:%M:%S"
)
insiders["end"] = pd.to_datetime(
    insiders["end"], format="%m/%d/%Y %H:%M:%S"
)

print("r4.2 insider records:", len(insiders))
print("Unique insider users:", insiders["user"].nunique())

r4.2 insider records: 70
Unique insider users: 70


In [5]:
results["day"] = pd.to_datetime(results["day"]).dt.normalize()
results["actual"] = 0

for _, row in insiders.iterrows():
    mask = (
        (results["user"] == row["user"]) &
        (results["day"] >= row["start"].normalize()) &
        (results["day"] <= row["end"].normalize())
    )

    results.loc[mask, "actual"] = 1

print(results["actual"].value_counts())

actual
0    329088
1      1364
Name: count, dtype: int64


In [6]:
y_true = results["actual"]
y_pred = results["anomaly"]

print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_true, y_pred, zero_division=0), 4))

Precision: 0.0061
Recall: 0.0704
F1-score: 0.0112


In [7]:
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["Normal", "Insider Scenario"],
    zero_division=0
))

Confusion Matrix:
[[313363  15725]
 [  1268     96]]

Classification Report:
                  precision    recall  f1-score   support

          Normal       1.00      0.95      0.97    329088
Insider Scenario       0.01      0.07      0.01      1364

        accuracy                           0.95    330452
       macro avg       0.50      0.51      0.49    330452
    weighted avg       0.99      0.95      0.97    330452



In [8]:
results.to_csv(
    "../data/processed/evaluation_results.csv",
    index=False
)

print("Saved evaluation_results.csv")

Saved evaluation_results.csv
